<a href="https://colab.research.google.com/github/Rakesh-Tegada/Brain-Guard-AI/blob/main/PPE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip -q install -U ultralytics gradio


In [3]:
from pathlib import Path
import json, shutil, time, os
import torch
from ultralytics import YOLO

PROJECT = Path('/content/ppe_context_project')
RUNS = PROJECT / 'runs'
MODELS = PROJECT / 'models'
MODELS.mkdir(parents=True, exist_ok=True)
DATA_URL = 'https://github.com/ultralytics/assets/releases/download/v0.0.0/construction-ppe.zip'
EPOCHS = 30
IMGSZ = 640
BATCH = 8 if torch.cuda.is_available() else 4
DEVICE = 0 if torch.cuda.is_available() else 'cpu'

free_gb = shutil.disk_usage('/content').free / (1024**3)
print(f'Runtime: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"}')
print(f'Free Colab disk: {free_gb:.1f} GB')
if free_gb < 2.0:
    raise RuntimeError('Less than 2 GB free. In Colab choose Runtime > Disconnect and delete runtime, reconnect, then run this notebook again.')
if not torch.cuda.is_available():
    print('WARNING: training on CPU will be slow. Select a GPU runtime if available before continuing.')

base = YOLO('yolo26n.pt')
train_results = base.train(
    data=DATA_URL,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    workers=2,
    cache=False,
    project=str(RUNS),
    name='ppe_context',
    exist_ok=True,
    plots=True,
    patience=8,
)

BEST = RUNS / 'ppe_context' / 'weights' / 'best.pt'
if not BEST.exists():
    raise FileNotFoundError(f'Training finished but best weights were not found at {BEST}')
FINAL_MODEL = MODELS / 'ppe_best.pt'
shutil.copy2(BEST, FINAL_MODEL)
print('Trained model saved:', FINAL_MODEL)


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Runtime: Tesla T4
Free Colab disk: 112.5 GB
Ultralytics 8.4.208 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=https://github.com/ultralytics/assets/releases/download/v0.0.0/construction-ppe.zip, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, e

In [4]:
# Evaluate the trained model on the dataset's held-out validation split.
# This is evaluation only; it does not train or change the weights.
model = YOLO(str(FINAL_MODEL))
val_metrics = model.val(
    data=DATA_URL,
    split='val',
    imgsz=IMGSZ,
    conf=0.001,
    device=DEVICE,
    plots=True,
    project=str(RUNS),
    name='ppe_validation',
    exist_ok=True,
)

metric_report = {
    'precision': float(val_metrics.box.mp),
    'recall': float(val_metrics.box.mr),
    'mAP50': float(val_metrics.box.map50),
    'mAP50_95': float(val_metrics.box.map),
    'inference_ms_per_image': float(val_metrics.speed.get('inference', 0.0)),
    'split': 'validation',
    'epochs_trained': EPOCHS,
    'image_size': IMGSZ,
    'model_file': str(FINAL_MODEL),
    'class_names': model.names,
}
print(json.dumps(metric_report, indent=2))
(PROJECT / 'validation_metrics.json').write_text(json.dumps(metric_report, indent=2), encoding='utf-8')
print('Validation plots and confusion matrix are saved under:', RUNS / 'ppe_validation')

Ultralytics 8.4.208 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n summary (fused): 121 layers, 2,392,408 parameters, 0 gradients, 5.4 GFLOPs
Found https://github.com/ultralytics/assets/releases/download/v0.0.0/construction-ppe.zip locally at construction-ppe.zip
WARNING ⚠️ Skipping construction-ppe.zip unzip as destination path /content/datasets/construction-ppe already exists.
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 4731.8±908.8 MB/s, size: 852.4 KB)
val: Scanning /content/datasets/construction-ppe/labels/val.cache... 200 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 200/200 52.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 13/13 2.4it/s 5.4s
                   all        200       2961      0.842       0.67      0.734      0.524
                helmet        146        294      0.932      0.772      0.835      0.624
                gloves        136        381      0.898

In [7]:
import cv2
import numpy as np
import gradio as gr
from collections import defaultdict
from tempfile import NamedTemporaryFile

# Canonical class names used by the rule layer. Unknown classes are left intact.
def canonical(name):
    s = str(name).strip().lower().replace('-', '_').replace(' ', '_')
    aliases = {
        'person': 'person', 'worker': 'person',
        'hardhat': 'helmet', 'hard_hat': 'helmet',
        'no_hardhat': 'no_helmet', 'nohelmet': 'no_helmet',
        'no_safety_vest': 'no_vest', 'safety_vest': 'vest',
    }
    return aliases.get(s, s)


def zone_pixels(width, height, left, top, right, bottom):
    xa, xb = sorted((float(left), float(right)))
    ya, yb = sorted((float(top), float(bottom)))
    return (int(width*xa), int(height*ya), int(width*xb), int(height*yb))


def assess_and_draw(result, frame_bgr, zone, state, confirm_frames=8, image_mode=False):
    dets = []
    if result.boxes is not None:
        for i, box in enumerate(result.boxes):
            label = canonical(result.names[int(box.cls.item())])
            coords = tuple(int(v) for v in box.xyxy[0].tolist())
            tid = int(box.id.item()) if box.id is not None else None
            dets.append((coords, label, tid))

    zx1, zy1, zx2, zy2 = zone
    people = [d for d in dets if d[1] == 'person']
    gear = [d for d in dets if d[1] in {'helmet','no_helmet','vest','no_vest'}]
    events = []

    for idx, (pbox, _, track_id) in enumerate(people):
        x1, y1, x2, y2 = pbox
        px, py = (x1+x2)/2, (y1+y2)/2
        in_zone = zx1 <= px <= zx2 and zy1 <= py <= zy2
        assigned = []
        for ebox, label, _ in gear:
            ex1, ey1, ex2, ey2 = ebox
            cx, cy = (ex1+ex2)/2, (ey1+ey2)/2
            if x1 <= cx <= x2 and y1 <= cy <= y2:
                assigned.append(label)
        missing = []
        if 'no_helmet' in assigned and 'helmet' not in assigned:
            missing.append('helmet')
        if 'no_vest' in assigned and 'vest' not in assigned:
            missing.append('vest')

        key = track_id if track_id is not None else f'image_{idx}'
        violation = in_zone and bool(missing)
        state['frames'][key] = state['frames'].get(key, 0) + 1 if violation else 0
        required = 1 if image_mode else confirm_frames
        if not violation or state['frames'][key] < required:
            continue
        now = time.time()
        if now - state['last'].get(key, 0) < 5:
            continue
        persistence_points = min(15, state['frames'][key] // max(1, required))
        score = min(100, 35 + 25*len(missing) + persistence_points)
        level = 'HIGH' if score >= 70 else 'MEDIUM' if score >= 40 else 'LOW'
        events.append({'worker_id': track_id if track_id is not None else 'untracked',
                       'missing_ppe': ', '.join(missing), 'risk_score': score,
                       'risk_level': level, 'zone': 'hazard zone'})
        state['last'][key] = now

    annotated = result.plot()
    cv2.rectangle(annotated, (zx1,zy1), (zx2,zy2), (0,165,255), 2)
    cv2.putText(annotated, 'HAZARD ZONE', (zx1+5, max(20,zy1-8)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,165,255), 2)
    if events:
        e = events[0]
        cv2.putText(annotated, f"{e['risk_level']} RISK: {e['missing_ppe']}", (12,30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,0,255), 2)
    return annotated, events


def analyze_image(image, left, top, right, bottom, confidence):
    if image is None:
        raise gr.Error('Upload an image first.')
    frame = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    result = model.predict(frame, imgsz=IMGSZ, conf=confidence, verbose=False)[0]
    zone = zone_pixels(frame.shape[1], frame.shape[0], left, top, right, bottom)
    annotated, events = assess_and_draw(result, frame, zone, {'frames':{}, 'last':{}}, image_mode=True)
    report = json.dumps(events, indent=2) if events else 'No explicit missing-PPE detection confirmed in the hazard zone.'
    return cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB), report


def analyze_video(video_path, left, top, right, bottom, confidence, confirm_frames):
    if not video_path:
        raise gr.Error('Upload a video first.')
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise gr.Error('Could not open that video file.')
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    ok, first = cap.read()
    if not ok:
        cap.release()
        raise gr.Error('The uploaded video contains no readable frames.')
    h, w = first.shape[:2]
    tmp = NamedTemporaryFile(suffix='.mp4', delete=False)
    output_path = tmp.name
    tmp.close()
    writer = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w,h))
    if not writer.isOpened():
        cap.release()
        raise gr.Error('Could not create the annotated output video.')

    state = {'frames':{}, 'last':{}}
    all_events = []
    frame = first
    frame_index = 0
    while True:
        zone = zone_pixels(w, h, left, top, right, bottom)
        result = model.track(frame, persist=True, tracker='bytetrack.yaml',
                             imgsz=IMGSZ, conf=confidence, verbose=False)[0]
        annotated, events = assess_and_draw(result, frame, zone, state, confirm_frames)
        all_events.extend(events)
        writer.write(annotated)
        frame_index += 1
        ok, frame = cap.read()
        if not ok:
            break
    cap.release()
    writer.release()
    summary = f'Processed {frame_index} frames; confirmed events: {len(all_events)}.'
    if all_events:
        summary += chr(10) + json.dumps(all_events[:100], indent=2)
    return output_path, summary


with gr.Blocks(title='Context-Aware PPE Safety Monitor') as demo:
    gr.Markdown('# Context-Aware PPE Safety Monitor | YOLO PPE detection + ByteTrack IDs + hazard-zone rule + persistence-based risk scoring.')
    with gr.Row():
        left = gr.Slider(0, 1, value=0.25, label='Zone left edge (fraction of width)')
        top = gr.Slider(0, 1, value=0.10, label='Zone top edge (fraction of height)')
        right = gr.Slider(0, 1, value=0.90, label='Zone right edge (fraction of width)')
        bottom = gr.Slider(0, 1, value=0.95, label='Zone bottom edge (fraction of height)')
    confidence = gr.Slider(0.05, 0.90, value=0.25, label='Detection confidence')
    confirm = gr.Slider(1, 60, value=8, step=1, label='Consecutive video frames to confirm a violation')
    with gr.Tab('Image'):
        image_in = gr.Image(type='numpy', label='Worksite image')
        image_out = gr.Image(type='numpy', label='Annotated result')
        image_report = gr.Textbox(label='Risk events')
        image_button = gr.Button('Analyze image')
        image_button.click(analyze_image, [image_in,left,top,right,bottom,confidence], [image_out,image_report])
    with gr.Tab('Video'):
        video_in = gr.Video(sources=['upload'], label='Worksite video')
        video_out = gr.Video(label='Annotated video')
        video_report = gr.Textbox(label='Risk events')
        video_button = gr.Button('Analyze video')
        video_button.click(analyze_video, [video_in,left,top,right,bottom,confidence,confirm], [video_out,video_report])
    gr.Markdown('The score is a rule-based demo indicator, not an injury probability. Missing-PPE classes are required; absence of a visible item is not treated as proof of non-compliance.')

demo.launch(share=True, debug=False)



Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://01ab2a07fed4ffb771.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
